# Topic: SQL Rolling Average (Moving Average)

## Definition (30-second explanation)
* A rolling average smooths out day-to-day fluctuations in a metric by averaging values over a sliding window of N periods.
* It helps reveal underlying trends that might be hidden by short-term noise.

## Why Interviewers Ask This
* Evaluates your mastery of advanced SQL window functions and the `ROWS BETWEEN` clause.
* Tests your ability to handle real-world time-series data and build dashboard-ready metrics.
* Checks if you understand the mathematical relationship between the window size and the `PRECEDING` value (N-day window = N-1 preceding).

## Core Concepts
* **Window Frame:** Uses `ROWS BETWEEN N PRECEDING AND CURRENT ROW` to define the sliding window.
* **Calculation:** For an N-day rolling average, you must use `(N-1) PRECEDING` (e.g., a 3-day average uses `2 PRECEDING`).
* **Early Rows:** SQL automatically handles early rows by averaging only the available data points (e.g., row 1 averages 1 value).

## When to Use
* Smoothing noisy metrics like daily revenue or page views.
* Building standard N-day (e.g., 7-day or 30-day) moving average charts for business dashboards.
* Detecting long-term trends in time-series analysis.

## Advantages
* Reduces the impact of outliers and daily seasonality (like weekend dips).
* `AVG() OVER(...)` computes the metric without requiring self-joins.

## Limitations
* If dates are missing from the dataset, `ROWS BETWEEN` will average the last N *rows*, not the last N *days*, which can skew results.
* Raw output can have many decimal places, requiring the `ROUND()` function to clean up.

## Common Comparisons
* **Rolling Average vs. Running Average:** Rolling average uses a fixed sliding window to smooth noise, whereas a running average grows cumulatively from the start of the dataset.

## Common Interview Traps
* **Off-by-one errors:** Writing `3 PRECEDING` for a 3-day average instead of `2 PRECEDING`.
* **Missing `ORDER BY`:** Forgetting `ORDER BY` inside the `OVER()` clause makes the window undefined.
* **Missing `PARTITION BY`:** Forgetting to separate the window per group (e.g., per product or region) when calculating grouped metrics.

## Python / SQL Syntax (if applicable)
```sql
SELECT 
    date,
    ROUND(AVG(metric) OVER (
        PARTITION BY category_id 
        ORDER BY date 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS rolling_7day_avg
FROM table_name;
```

## Important Formula (if applicable)
* **N-day window** = `(N-1) PRECEDING AND CURRENT ROW`.

## 45-Second Interview Answer
"To calculate a rolling average in SQL, I use the `AVG()` window function combined with an `ORDER BY` date clause and a `ROWS BETWEEN` frame. The key trick is that for an N-day moving average, the frame should be `(N-1) PRECEDING AND CURRENT ROW`. For example, a 7-day rolling average uses `6 PRECEDING`. I always make sure to include a `PARTITION BY` if the metric needs to be grouped by product or region, and I wrap it in a `ROUND()` function to clean up the decimal output."

## Example Questions and Answers:

### Q1. Compute a 7-day rolling average of page views from the web_analytics table.

**Ideal Interview Answer:**
```sql
SELECT
    view_date,
    page_views,
    ROUND(AVG(page_views) OVER (
        ORDER BY view_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS rolling_7day_views
FROM web_analytics;
```

**Common Mistakes Candidates Make:**
* Using `ROWS BETWEEN 7 PRECEDING` instead of `6 PRECEDING`.
* Forgetting the `ORDER BY view_date` inside the window function.

**One Likely Interviewer Follow-up:**
"What happens to the first 3 days of data in this query? Does it return NULL?" 
*(Answer: No, SQL automatically averages only the available rows for those early periods. Row 1 averages 1 value, row 2 averages 2 values, etc.)*

### Q2. Calculate a 30-day rolling average of daily revenue, partitioned by region.

**Ideal Interview Answer:**
```sql
SELECT
    region,
    revenue_date,
    daily_revenue,
    ROUND(AVG(daily_revenue) OVER (
        PARTITION BY region
        ORDER BY revenue_date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) AS rolling_30day_rev
FROM regional_sales
ORDER BY region, revenue_date;
```

**Common Mistakes Candidates Make:**
* Missing the `PARTITION BY region` clause, causing the rolling average to bleed across different regions.

**One Likely Interviewer Follow-up:**
"How would the results change if a region had no sales recorded (missing rows) for several days in the middle of the month?"
*(Answer: `ROWS BETWEEN` counts physical rows, not calendar days. A gap in dates means older, irrelevant data points get pulled into the 30-row window. You would need to generate a date series and `LEFT JOIN` the sales data to ensure every calendar day is represented.)*

### Q3. Find all dates where the 7-day rolling average exceeded 500 units sold.

**Ideal Interview Answer:**
```sql
WITH RollingStats AS (
    SELECT
        sale_date,
        ROUND(AVG(units_sold) OVER (
            ORDER BY sale_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 2) AS rolling_7day_avg
    FROM daily_sales
)
SELECT sale_date, rolling_7day_avg
FROM RollingStats
WHERE rolling_7day_avg > 500;
```

**Common Mistakes Candidates Make:**
* Trying to use the window function directly in the `WHERE` clause, which SQL does not allow. You must use a CTE or subquery first.

**One Likely Interviewer Follow-up:**
"Could you modify this to only flag dates where the rolling average was > 500 for three consecutive days?"

### Q4. Compare the daily value against its 3-day rolling average to spot anomalies.

**Ideal Interview Answer:**
```sql
SELECT
    metric_date,
    daily_value,
    ROUND(AVG(daily_value) OVER (
        ORDER BY metric_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS rolling_avg_3day,
    daily_value - ROUND(AVG(daily_value) OVER (
        ORDER BY metric_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS difference_from_avg
FROM daily_metric
ORDER BY metric_date;
```

**Common Mistakes Candidates Make:**
* Not wrapping the window function calculation properly when performing the subtraction, or doing it in a way that causes integer division issues.

**One Likely Interviewer Follow-up:**
"How would you calculate a percentage deviation from the rolling average instead of an absolute difference?"

## Practice Questions:

In [1]:
import sqlite3
import pandas as pd

conn= sqlite3.connect('/home/shail/interview-prep/01_SQL/oracle_hr.db')

### Q1: 
**You are calculating a 3-day rolling average for daily sales. However, the store is occasionally closed, meaning some dates are entirely missing from the database.**

- Schema:
```sql
CREATE TABLE daily_sales (
    sale_date DATE,
    revenue INT
);

INSERT INTO daily_sales (sale_date, revenue) VALUES 
('2026-07-01', 100),
('2026-07-02', 200),
-- Store closed on July 3rd (missing row)
('2026-07-04', 300),
('2026-07-05', 400);
```

- Question: If you simply use ROWS BETWEEN 2 PRECEDING AND CURRENT ROW on this data, the 3-day average for 2026-07-04 will incorrectly include 2026-07-01 because it looks at the last 3 rows. How would you write a SQL query (in SQLite or MySQL) to accurately calculate a true 3-calendar-day rolling average that accounts for these missing date gaps?

*Answer:*
```sql
SELECT
    sale_date,
    revenue,
    ROUND(AVG(revenue) OVER (
        ORDER BY sale_date
        RANGE BETWEEN INTERVAL 2 DAY PRECEDING AND CURRENT ROW
    ), 2) AS rolling_3day_avg
FROM daily_sales;
```

**Interview Tips:**
* **ROWS vs. RANGE:** `ROWS BETWEEN` strictly counts the physical rows (ignoring the actual date values). `RANGE BETWEEN` evaluates the logical value of the `ORDER BY` column. 
* By using `RANGE BETWEEN INTERVAL 2 DAY PRECEDING`, SQL looks for records that logically fall within the last 48 hours, correctly skipping the gap.
* **Dialect Warning:** `INTERVAL` window framing works in MySQL, PostgreSQL, and BigQuery. It does *not* work cleanly in SQL Server or SQLite without converting dates to integers first.

### Q2: Business Thinking & System Design

Scenario:
Your RANGE query perfectly calculates the correct rolling average for the dates that do exist in the table. However, the business team looks at your dashboard and complains:

"Where is July 3rd? We need every single day of the month visible on the chart. If we were closed, it should show $0 for daily revenue, and the rolling average should still calculate properly for that day."

Question:
Since July 3rd does not exist in the daily_sales table at all, how would you architect your SQL approach to fulfill the business team's requirement? (You do not need to write the exact SQL code, just explain the conceptual steps/architecture you would use).

*Answer:*
**(Conceptual):**
1. **Generate a Date Series:** Create a sequence of all calendar dates from the minimum to the maximum date using a Recursive CTE (or query a pre-existing Calendar/Date dimension table).
2. **Left Join:** `LEFT JOIN` this complete date series to the `daily_sales` table.
3. **Handle Nulls:** Use `COALESCE(revenue, 0)` to replace the `NULL` revenue on closed days with `$0`.
4. **Calculate:** Apply the standard `AVG() OVER (ROWS BETWEEN...)` rolling average window function on the newly joined, gap-less dataset.

* **MySQL 8.0+ / SQLite Date Series Syntax (Recursive CTE):**
```sql
WITH RECURSIVE date_series AS (
    -- Anchor: Start Date
    SELECT MIN(sale_date) AS dte FROM daily_sales 
    UNION ALL
    -- Recursive Step: Add 1 day until Max Date
    SELECT DATE(dte, '+1 day') 
    FROM date_series 
    WHERE dte < (SELECT MAX(sale_date) FROM daily_sales)
)
SELECT 
    d.dte AS sale_date,
    COALESCE(s.revenue, 0) AS daily_revenue,
    ROUND(AVG(COALESCE(s.revenue, 0)) OVER (
        ORDER BY d.dte
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS rolling_3day_avg
FROM date_series d
LEFT JOIN daily_sales s ON d.dte = s.sale_date;
```
* Using a date series guarantees that `ROWS BETWEEN N PRECEDING` will accurately represent `N days`, completely eliminating the need for the `RANGE` keyword.

### Q3: Advanced Business Logic & Data Imputation

Scenario:
The HR Director wants to analyze hiring velocity. They want a report showing the 7-day rolling average of total starting salaries for new hires, mapped out day-by-day for a specific period in January 2001.

Since we don't hire people every day, there are gaps in the HIRE_DATE column. The HR Director explicitly requested that days with zero hires must appear in the report as $0 and be mathematically factored into the rolling average correctly.

- Your Task: Write a SQL query (SQLite/MySQL compatible) that executes the following steps:
1. Uses a Recursive CTE to generate a continuous date series strictly between '2001-01-10' and '2001-01-25'.
2. LEFT JOINs the EMPLOYEES table to this date series.
3. Calculates the total SALARY hired on each day (remembering to treat NULL as 0).
4. Computes the 7-day rolling average (the current day + 6 preceding days) of that daily salary total.

In [10]:
pd.read_sql_query(sql= """
WITH RECURSIVE date_series AS (
    SELECT '2011-01-13' AS report_date
    UNION ALL
    SELECT DATE(report_date, '+1 day')
    FROM date_series
    WHERE report_date < '2011-01-31'
),
DailyTotals AS (
    -- STEP 1: Aggregate to ensure exactly ONE row per day
    SELECT 
        d.report_date,
        COALESCE(SUM(e.salary), 0) AS total_daily_salary
    FROM date_series d
    LEFT JOIN employees e ON d.report_date = e.hire_date
    GROUP BY d.report_date
)
-- STEP 2: Apply the rolling average to the aggregated data
SELECT 
    report_date,
    total_daily_salary,
    ROUND(AVG(total_daily_salary) OVER (
        ORDER BY report_date 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS rolling_7day_avg
FROM DailyTotals
ORDER BY report_date;
""", con= conn)

,report_date,total_daily_salary,rolling_7day_avg
0,2011-01-13,17000.0,17000.00
1,2011-01-14,0.0,8500.00
2,2011-01-15,0.0,5666.67
3,2011-01-16,0.0,4250.00
4,2011-01-17,0.0,3400.00
5,2011-01-18,0.0,2833.33
6,2011-01-19,0.0,2428.57
7,2011-01-20,0.0,0.00
8,2011-01-21,0.0,0.00
9,2011-01-22,0.0,0.00


**Interview Tips:**
* **The Granularity Trap:** If your window frame expects "1 row = 1 day" (e.g., `ROWS BETWEEN 6 PRECEDING`), you must guarantee your dataset is strictly aggregated to one row per day *before* applying the window function. 
* If you skip the `GROUP BY`, days with multiple events will spawn multiple rows, breaking the chronological math of your `ROWS BETWEEN` frame.